# LAB S04: extracción de información y PLN

Cada sección usa el archivo de `mi_pipeline` que se escribe en su paso de la guía; el cuaderno solo importa y ejecuta. Se trabaja con documentos ficticios y con el texto que leyó el OCR en la sesión anterior; ningún documento se sube a servicios en la nube.

## 0. Dónde están los datos de práctica

In [ ]:
# Celda estándar de localización de datos (no se modifica)
import os
from pathlib import Path


def localizar_carpeta(nombre="02_Datos_Practica", max_niveles=6, obligatorio=True):
    """Busca `nombre` en la carpeta actual y en sus carpetas superiores."""
    actual = Path.cwd().resolve()
    for carpeta in [actual, *actual.parents][: max_niveles + 1]:
        candidata = carpeta / nombre
        if candidata.is_dir():
            return candidata
    # Respaldo: carpetas del curso en la unidad del sistema (VM de laboratorio)
    curso = Path(os.environ.get("SystemDrive", "C:") + os.sep) / "curso"
    respaldo = [curso / "paquete" / nombre]
    if nombre == "02_Datos_Practica":
        respaldo.insert(0, curso / "datos")
    for candidata in respaldo:
        if candidata.is_dir():
            return candidata
    if obligatorio:
        raise FileNotFoundError(
            f"No se encuentra la carpeta '{nombre}'. El cuaderno se abre desde su carpeta "
            "dentro del repositorio del curso, sin moverlo de lugar."
        )
    return None


DATOS = localizar_carpeta("02_Datos_Practica")
print("Datos de práctica encontrados:", DATOS.name)
print("Subcarpetas:", sorted(p.name for p in DATOS.iterdir() if p.is_dir()))


## Paso 0: Conexión con `mi_pipeline` y rama de trabajo (C0)

In [ ]:
# Conexión de mi_pipeline y de las herramientas del LAB (no se modifica)
%load_ext autoreload
%autoreload 2
import email
import json
import os
import re
import subprocess
import sys
import tempfile
import time
from email import policy

import pandas as pd

os.environ.setdefault("HF_HUB_OFFLINE", "1")   # los modelos ya están en la VM: no se descarga nada
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 50)


def localizar_raiz(max_niveles=6):
    """Carpeta del repositorio clonado que contiene mi_pipeline."""
    if os.environ.get("MI_PIPELINE_RAIZ"):
        return Path(os.environ["MI_PIPELINE_RAIZ"])
    actual = Path.cwd().resolve()
    for carpeta in [actual, *actual.parents][: max_niveles + 1]:
        if (carpeta / "mi_pipeline" / "__init__.py").is_file():
            return carpeta
    clon = Path(os.environ.get("SystemDrive", "C:") + os.sep) / "curso" / "trabajo" / "entorno-extraccion-documental"
    if (clon / "mi_pipeline" / "__init__.py").is_file():
        return clon
    raise FileNotFoundError("No se encuentra la carpeta mi_pipeline: se creó en la sesión anterior.")


RAIZ = localizar_raiz()
sys.path.insert(0, str(RAIZ))
SALIDAS = Path("salidas")
SALIDAS.mkdir(exist_ok=True)
PRECALCULADAS = DATOS / "11_Salidas_Precalculadas"
OCR_GUARDADO = PRECALCULADAS / "ocr_s04"
CONTROLES = []
INICIO = time.perf_counter()


def correr_pruebas():
    """Corre las pruebas de mi_pipeline con pytest, muestra el resumen y dice si todas pasaron."""
    corrida = subprocess.run(
        [sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider", str(RAIZ / "mi_pipeline" / "tests")],
        capture_output=True, text=True, cwd=str(RAIZ), timeout=600, encoding="utf-8", errors="replace",
    )
    resumen = [linea for linea in corrida.stdout.strip().splitlines() if " passed" in linea or " failed" in linea or "error" in linea]
    print(re.sub(r" in [\d.]+s.*", "", resumen[-1]) if resumen else corrida.stdout[-500:])
    return corrida.returncode == 0


def leer_correos_con_ocr(receta, psm, enderezar):
    """Cuerpo de cada correo y texto del OCR de su adjunto, leído con `ejecutar` de S03 y los ajustes indicados."""
    from mi_pipeline.pipeline import cargar_configuracion, ejecutar

    cfg = cargar_configuracion()
    cfg["ocr"]["psm"] = psm
    correos = {}
    with tempfile.TemporaryDirectory() as carpeta:
        for eml in sorted((DATOS / "06_Reclamos").glob("REC-*.eml")):
            mensaje = email.message_from_binary_file(eml.open("rb"), policy=policy.default)
            adjunto = next(mensaje.iter_attachments())
            archivo = Path(carpeta) / adjunto.get_filename()
            archivo.write_bytes(adjunto.get_content())
            ocr = ejecutar(archivo, cfg, receta=receta, enderezar=enderezar)["texto"]
            correos[eml.stem] = {"cuerpo": mensaje.get_body(preferencelist=("plain",)).get_content().strip(), "ocr": ocr}
    return correos


def cargar_lote():
    """Los 60 reclamos: asunto, cuerpo, texto del OCR del adjunto (ya guardado), texto completo y respuestas correctas."""
    from mi_pipeline.extraccion.texto import cargar_texto, leer_correo

    lote = []
    for eml in sorted((DATOS / "06_Reclamos").glob("REC-*.eml")):
        correo = leer_correo(eml)
        ocr = cargar_texto(OCR_GUARDADO / "reclamos" / f"{eml.stem}.json")
        correcto = json.loads((DATOS / "10_Ground_Truth" / f"{eml.stem}.json").read_text(encoding="utf-8"))["campos"]
        lote.append({"id": eml.stem, "asunto": correo["asunto"], "cuerpo": correo["cuerpo"], "ocr": ocr,
                     "texto": f"De: {correo['remitente']}\n{correo['cuerpo']}\n{ocr}", "correcto": correcto})
    return lote


print("Carpeta del repositorio:", RAIZ.name, "| reclamos:", len(list((DATOS / "06_Reclamos").glob("REC-*.eml"))))


In [ ]:
# ✔ Control C0: la rama de trabajo
rama = subprocess.run(["git", "branch", "--show-current"], cwd=RAIZ, capture_output=True, text=True).stdout.strip()
assert rama == "trabajo-s04", f"C0: la rama actual es '{rama}'; la guía indica crear trabajo-s04 desde trabajo-s03."
CONTROLES.append("C0")
print("✔ C0 superado: rama de trabajo", rama)

## Reto: extraer DNI, fecha y monto de 60 correos con `re.findall`

In [ ]:
# Reto: DNI, fecha y monto de los 60 correos con re.findall, sobre el texto del adjunto que lee el OCR de S03
ajustes_s03 = leer_correos_con_ocr(receta="mediana", psm=6, enderezar=True)      # valores del YAML de S03 (hechos para oficios)
ajustes_adjunto = leer_correos_con_ocr(receta="crudo", psm=3, enderezar=False)   # valores que sirven para este adjunto


def contar(correos, campo, patron):
    """Cuántos de los 60 correos traen el patrón en el campo indicado (cuerpo u ocr)."""
    return sum(bool(re.findall(patron, c[campo])) for c in correos.values())


reto = pd.DataFrame({"OCR con ajustes de S03": [contar(ajustes_s03, "ocr", r"99\d{6}")],
                     "OCR con ajustes del adjunto": [contar(ajustes_adjunto, "ocr", r"99\d{6}")]}, index=["DNI leído en el adjunto"])
reto.loc["fecha completa en el adjunto"] = [contar(c, "ocr", r"\d{2}/\d{2}/\d{4}") for c in (ajustes_s03, ajustes_adjunto)]
reto.loc["monto con S/ en el cuerpo"] = [contar(ajustes_adjunto, "cuerpo", r"S/\s*[\d,]+")] * 2
print(reto)

In [ ]:
# Causas de las fallas: el participante escribe al menos dos, entre comillas y separadas por coma
causas = []
if len(causas) >= 2:
    print(f"✔ Reto superado: anotó {len(causas)} causas ->", "; ".join(causas))
else:
    print("✘ Falta anotar al menos dos causas en la lista `causas` y volver a correr esta celda")

## Paso 1: `extraccion/texto.py` (E1, C1 y C2)

In [ ]:
# E1: el texto de 5 correos y de sus adjuntos, y su análisis con spaCy (C1)
from mi_pipeline.extraccion.texto import analizar, cargar_texto, formas_base

IDS = ["REC-0001", "REC-0002", "REC-0003", "REC-0004", "REC-0005"]
for id_ in IDS:
    cuerpo = cargar_texto(DATOS / "06_Reclamos" / f"{id_}.eml")
    ocr_json = OCR_GUARDADO / "reclamos" / f"{id_}.json"
    correcto = json.loads((DATOS / "10_Ground_Truth" / f"{id_}.json").read_text(encoding="utf-8"))["texto_completo"]
    # ✔ Control C1
    assert correcto.startswith(cuerpo), f"C1: el cuerpo de {id_} no coincide con el texto correcto."
    assert cargar_texto(ocr_json) == json.loads(ocr_json.read_text(encoding="utf-8"))["texto"].strip(), f"C1: el texto de {id_} no coincide con el del OCR."
CONTROLES.append("C1")
doc = analizar(cargar_texto(DATOS / "06_Reclamos" / "REC-0001.eml"))
print(pd.DataFrame([{"token": t.text, "forma base": t.lemma_, "clase": t.pos_} for t in doc][:12]).T)
print("Entidades que marca spaCy:", [(e.text, e.label_) for e in doc.ents])
print(f"✔ C1 superado: el texto cargado de {len(IDS)} correos y sus adjuntos coincide con el correcto y con el del OCR")

In [ ]:
# E1: formas base (lemas), control C2
print("Formas base:", formas_base("Los cobros indebidos del banco"))
# ✔ Control C2
assert formas_base("cobro")[0] == formas_base("cobros")[0], "C2: «cobro» y «cobros» deben compartir forma base."
CONTROLES.append("C2")
print("✔ C2 superado: «cobro» y «cobros» comparten la forma base", formas_base("cobros")[0])
print("Un error del OCR se hereda:", [t.text for t in analizar(cargar_texto(OCR_GUARDADO / "reclamos" / "REC-0005.json"))][:14])

## Paso 2: `extraccion/reglas.py` (E2, C3)

In [ ]:
# E2: reglas sobre el texto de los 60 reclamos (remitente, cuerpo y OCR del adjunto), control C3
from mi_pipeline.extraccion.reglas import extraer_dni, extraer_fecha

lote = cargar_lote()
pred_dni = [extraer_dni(r["texto"]) for r in lote]
aciertos = sum(p == r["correcto"]["dni"] for p, r in zip(pred_dni, lote))
precision = aciertos / sum(p is not None for p in pred_dni)
recall = aciertos / len(lote)
f1_dni = 2 * precision * recall / (precision + recall)
print(f"DNI: {aciertos} de {len(lote)} correctos, precisión {precision:.2f}, recall {recall:.2f}, F1 {f1_dni:.2f}")
print("Fecha completa correcta en", sum(extraer_fecha(r["texto"]) == r["correcto"]["fecha"] for r in lote), "de", len(lote), "correos")
# ✔ Control C3
assert f1_dni == 1.0, "C3: el F1 del DNI debe ser 1,00."
CONTROLES.append("C3")
print("✔ C3 superado: DNI con F1 =", round(f1_dni, 2))

In [ ]:
# E2: RUC con módulo 11 y número de oficio en los oficios escaneados que leyó S03
from mi_pipeline.extraccion.reglas import extraer_numero_oficio, extraer_ruc

filas = []
for ruta in sorted((OCR_GUARDADO / "oficios").glob("OFI-*.json")):
    texto = cargar_texto(ruta)
    campos = json.loads((DATOS / "10_Ground_Truth" / f"{ruta.stem}.json").read_text(encoding="utf-8"))["campos"]
    ruc, numero = extraer_ruc(texto), extraer_numero_oficio(texto)
    ruc_malo = campos["ruc"][:-1] + str((int(campos["ruc"][-1]) + 1) % 10)
    filas.append({"oficio": ruta.stem, "RUC": ruc, "RUC correcto": ruc == campos["ruc"], "con un dígito cambiado": extraer_ruc(texto.replace(campos["ruc"], ruc_malo)),
                  "número": numero, "número correcto": numero == campos["numero_oficio"]})
tabla_oficios = pd.DataFrame(filas).set_index("oficio")
print(tabla_oficios)
assert tabla_oficios["con un dígito cambiado"].isna().all(), "Un RUC con el dígito verificador cambiado no debe aceptarse."

In [ ]:
# E2: EntityRuler de spaCy: patrones propios (DNI, teléfono y producto) antes del NER del modelo
from mi_pipeline.extraccion.reglas import modelo_con_reglas

nlp_reglas = modelo_con_reglas()
doc = nlp_reglas(lote[0]["cuerpo"])
print([(e.text, e.label_) for e in doc.ents])
assert {e.label_ for e in doc.ents} >= {"DNI"}, "El ruler debe marcar el DNI."

## Paso 3: `extraccion/montos.py` (E2, C4) y pruebas

In [ ]:
# E2: monto en cifras y monto en palabras, control C4
from mi_pipeline.extraccion.montos import extraer_monto, monto_a_numero

print("«dos mil seiscientos cuarenta y tres soles» ->", monto_a_numero("dos mil seiscientos cuarenta y tres soles"))
correctos_monto = [r["correcto"]["monto_reclamado"] for r in lote]
solo_cifras = [extraer_monto(r["cuerpo"], palabras=False) for r in lote]
con_palabras = [extraer_monto(r["cuerpo"]) for r in lote]
recall_cifras = sum(m == c for m, c in zip(solo_cifras, correctos_monto)) / len(lote)
recall_palabras = sum(m == c for m, c in zip(con_palabras, correctos_monto)) / len(lote)
print(f"Recall del monto en el cuerpo: solo cifras {recall_cifras:.2f}, con conversión de palabras {recall_palabras:.2f}")
# ✔ Control C4
assert monto_a_numero("dos mil seiscientos cuarenta y tres") == 2643, "C4: el monto en palabras debe convertirse a 2643."
assert recall_cifras < 0.5 and recall_palabras >= 0.95, "C4: la conversión de palabras debe subir el recall de menos de 0,50 a más de 0,95."
CONTROLES.append("C4")
print("✔ C4 superado: el monto en palabras se convierte a número")

In [ ]:
# ✔ Pruebas automáticas de los pasos 1 a 3
assert correr_pruebas(), "Alguna prueba falla: se lee el mensaje de pytest y se corrige el archivo que indica."
print("✔ Las pruebas de mi_pipeline pasan")

## Pasos 4 y 5: `calidad/f1.py` y `extraccion/ner.py` (E3, C5 y C6)

In [ ]:
# E3: F1 por campo con la función propia, enfoque de reglas (C5)
from mi_pipeline.calidad.f1 import campos_bajo_umbral, f1_por_campo
from mi_pipeline.extraccion.reglas import (extraer_correo_electronico, extraer_entidad, extraer_motivo, extraer_nombre,
                                           extraer_producto, extraer_telefono)

EXTRACTORES = {"dni": extraer_dni, "nombre": extraer_nombre, "monto_reclamado": extraer_monto, "entidad": extraer_entidad,
               "telefono": extraer_telefono, "correo": extraer_correo_electronico, "producto": extraer_producto,
               "fecha": extraer_fecha}
correctos = [r["correcto"] for r in lote]
pred_reglas = [{campo: funcion(r["texto"]) for campo, funcion in EXTRACTORES.items()} | {"motivo": extraer_motivo(r["ocr"])} for r in lote]
F1 = {"reglas": {campo: m["f1"] for campo, m in f1_por_campo(pred_reglas, correctos, [*EXTRACTORES, "motivo"]).items()}}
print(pd.Series(F1["reglas"], name="F1 reglas"))
# ✔ Control C5 (primera parte): la función propia reproduce el cálculo a mano de C3
assert F1["reglas"]["dni"] == round(f1_dni, 3), "C5: la función f1_por_campo debe dar el mismo F1 del DNI que el cálculo a mano."

In [ ]:
# E3: NER con spaCy y con Transformers, y F1 de nombre y entidad (C5 y C6)
from mi_pipeline.extraccion.ner import entidades_a_campos, ner_spacy, ner_transformers, resolver_dispositivo

t0 = time.perf_counter()
ents_spacy = [ner_spacy(r["texto"]) for r in lote]
segundos_spacy = time.perf_counter() - t0
t0 = time.perf_counter()
ents_trf = [ner_transformers(r["texto"], precalculado=PRECALCULADAS / "ner" / f"{r['id']}.json") for r in lote]
segundos_trf = time.perf_counter() - t0
pred_spacy = [entidades_a_campos(e) for e in ents_spacy]
pred_trf = [entidades_a_campos(e) for e in ents_trf]
F1["spaCy"] = {campo: m["f1"] for campo, m in f1_por_campo(pred_spacy, correctos, ["nombre", "entidad"]).items()}
F1["Transformers"] = {campo: m["f1"] for campo, m in f1_por_campo(pred_trf, correctos, ["nombre", "entidad"]).items()}
print(pd.DataFrame(F1).loc[["nombre", "entidad"]])
print(f"Dispositivo: {resolver_dispositivo()}; 60 textos: spaCy {segundos_spacy:.1f} s, Transformers {segundos_trf:.1f} s (la primera llamada incluye la carga del modelo)")
# ✔ Controles C5 y C6
assert set(F1["spaCy"]) == set(F1["Transformers"]) == {"nombre", "entidad"}, "C5: el F1 se calcula por campo."
CONTROLES.append("C5")
assert F1["Transformers"]["nombre"] > F1["spaCy"]["nombre"] and F1["Transformers"]["entidad"] > F1["spaCy"]["entidad"], "C6: Transformers debe superar a spaCy en nombre y entidad."
CONTROLES.append("C6")
print("✔ C5 y C6 superados: Transformers supera a spaCy en nombre y entidad")

In [ ]:
# Lote público real (opcional): empresa y fecha de presentación de 7 políticas de dividendos frente a su ficha; no se muestran personas ni textos
from mi_pipeline.extraccion.lote_real import campos_desde_ner, localizar_lote, medir_lote_real

sintetico = {"reglas": {"empresa": F1["reglas"]["entidad"], "fecha": F1["reglas"]["fecha"]},
             "spaCy": {"empresa": F1["spaCy"]["entidad"], "fecha": 0.0},
             "Transformers": {"empresa": F1["Transformers"]["entidad"],
                              "fecha": f1_por_campo([{"fecha": campos_desde_ner(e)["fecha"]} for e in ents_trf], correctos, ["fecha"])["fecha"]["f1"]}}
real = medir_lote_real(localizar_lote())
if real:
    comparacion = pd.concat({"60 reclamos (sintético)": pd.DataFrame(sintetico), f"{len(real['documentos'])} políticas (real)": pd.DataFrame(real["f1"])}, axis=1)
    print(comparacion.round(3))
    print("Caída del F1 al pasar a documentos reales:")
    print((pd.DataFrame(real["f1"]) - pd.DataFrame(sintetico)).round(3))
    assert set(real["f1"]) == {"reglas", "spaCy", "Transformers"}, "El F1 real se mide para los tres enfoques."
else:
    print("Sección opcional omitida: en esta máquina no hay lote público real (carpeta curso/datos/14_Publico_Real).")

## Paso 6: `clasificacion/motivo.py` (E4, C7 y C8)

In [ ]:
# E4: motivo del reclamo con TF-IDF y regresión logística, solo con el cuerpo (C7)
from mi_pipeline.clasificacion.motivo import entrenar, f1_macro, validar_cruzado, zero_shot

cuerpos = [r["cuerpo"] for r in lote]
motivos = [r["correcto"]["motivo"] for r in lote]
cv = validar_cruzado(cuerpos, motivos)
print(f"TF-IDF + regresión logística, 5 pliegues: F1 macro {cv['f1_macro']:.3f}, exactitud {cv['exactitud']:.3f}")
print(pd.DataFrame(cv["matriz"], index=cv["clases"], columns=cv["clases"]))
con_asunto = validar_cruzado([r["asunto"] + " " + r["cuerpo"] for r in lote], motivos)
print(f"Con el asunto (trampa, el asunto trae la clase): F1 macro {con_asunto['f1_macro']:.3f}")
F1["TF-IDF"] = {"motivo": f1_por_campo([{"motivo": p} for p in cv["predicciones"]], correctos, ["motivo"])["motivo"]["f1"]}
# ✔ Control C7
assert cv["f1_macro"] > 0.8, "C7: el F1 macro de TF-IDF debe superar 0,80."
CONTROLES.append("C7")
print("✔ C7 superado: F1 macro de TF-IDF =", cv["f1_macro"])

In [ ]:
# E4: zero-shot sin ejemplos de entrenamiento y comparación con TF-IDF (C8)
etiquetas = sorted(set(motivos))
t0 = time.perf_counter()
salidas_zs = zero_shot(cuerpos, etiquetas, precalculado=[PRECALCULADAS / "zero_shot" / f"{r['id']}.json" for r in lote])
segundos_zs = time.perf_counter() - t0
pred_zs = [s["etiqueta"] for s in salidas_zs]
f1_zs = f1_macro(motivos, pred_zs)
F1["zero-shot"] = {"motivo": f1_por_campo([{"motivo": p} for p in pred_zs], correctos, ["motivo"])["motivo"]["f1"]}
print(f"Zero-shot: F1 macro {f1_zs:.3f} frente a {cv['f1_macro']:.3f} de TF-IDF; {segundos_zs:.1f} s para 60 correos ({resolver_dispositivo()})")
print(pd.DataFrame({"correcto": motivos, "zero-shot": pred_zs, "TF-IDF": cv["predicciones"]}).head(6))
# ✔ Control C8
assert cv["f1_macro"] > f1_zs, "C8: con 60 ejemplos TF-IDF debe superar al zero-shot."
CONTROLES.append("C8")
print("✔ C8 superado: TF-IDF", cv["f1_macro"], "frente a zero-shot", f1_zs)

## Paso 7: ensamblado en `pipeline.py` y caso (C9)

In [ ]:
# Caso (avance 4): el pipeline de extracción completo sobre los 60 correos, tabla de F1 y JSON por correo para S05 (C9)
from mi_pipeline.pipeline import cargar_configuracion, extraer, extraer_lote

CFG = cargar_configuracion()
modelo = entrenar(cuerpos, motivos)      # entrenado con el lote: sirve para la confianza y la ruta, no para medir el motivo
resultados = extraer_lote(DATOS / "06_Reclamos", CFG, modelo, OCR_GUARDADO / "reclamos")
pred_pipeline = [{**r["campos"], "motivo": p} for r, p in zip(resultados, cv["predicciones"])]   # motivo medido con validación cruzada
F1["pipeline"] = {campo: m["f1"] for campo, m in f1_por_campo(pred_pipeline, correctos, [*EXTRACTORES, "motivo"]).items()}
tabla_f1 = pd.DataFrame(F1).reindex([*EXTRACTORES, "motivo"]).round(3)
print(tabla_f1)
bajo = campos_bajo_umbral({c: {"f1": f} for c, f in F1["pipeline"].items()}, CFG["calidad"]["umbral_f1"])
print(f"Campos bajo el umbral de F1 ({CFG['calidad']['umbral_f1']}):", bajo)
rutas = pd.Series([r["ruta"] for r in resultados]).value_counts()
print("Ruta de los correos:", rutas.to_dict())
destino = SALIDAS / "extraccion"
destino.mkdir(exist_ok=True)
for r in resultados:
    (destino / f"{r['id']}.json").write_text(json.dumps(r, ensure_ascii=False, indent=1), encoding="utf-8")
tabla_f1.to_csv(SALIDAS / "f1_por_campo_y_enfoque.csv", encoding="utf-8")
en_vivo = extraer(DATOS / "06_Reclamos" / "REC-0001.eml", CFG, modelo)     # una corrida con el OCR de S03 en vivo
LOTE_REAL = localizar_lote(CFG["extraccion"]["carpeta_lote_real"])
real = medir_lote_real(LOTE_REAL, CFG["extraccion"]["dispositivo"])        # el mismo método sobre las 7 políticas reales
if real:
    diferencia = (pd.DataFrame(real["f1"]) - pd.DataFrame(sintetico)).round(3)
    print("Caída del F1 de empresa y fecha (real menos sintético):")
    print(diferencia)
    diferencia.to_csv(SALIDAS / "caida_f1_real.csv", encoding="utf-8")
else:
    print("Documentos reales omitidos: no hay lote público real en esta máquina.")
# ✔ Control C9
assert len(resultados) == 60 and not any("error" in r for r in resultados), "C9: los 60 correos deben procesarse sin error."
assert en_vivo["campos"] == resultados[0]["campos"], "C9: el OCR en vivo debe dar los mismos campos que el OCR guardado."
assert len(bajo) <= 1, "C9: como máximo un campo puede quedar bajo el umbral de F1."
CONTROLES.append("C9")
print(f"✔ C9 superado: 60 correos procesados, {len(bajo)} campo(s) bajo el umbral, JSON por correo en", destino)

## Q5: un campo que no llega al umbral

In [ ]:
# Campo bajo el umbral: el teléfono (pregunta Q5)
telefono = f1_por_campo(pred_reglas, correctos, ["telefono"])["telefono"]
traen = sum(p["telefono"] is not None for p in pred_reglas)
print(f"Teléfono: precisión {telefono['precision']:.2f}, recall {telefono['recall']:.2f}, F1 {telefono['f1']:.2f}; {traen} de {len(lote)} correos lo traen; umbral de F1 {CFG['calidad']['umbral_f1']}")
print("Con precisión 1,00, el F1 máximo posible es", round(2 * (traen / len(lote)) / (1 + traen / len(lote)), 2), ": lo que el correo no trae no lo recupera ninguna regla ni ningún modelo.")

## Reto extra (opcional)

In [ ]:
# Reto extra (opcional): otra frase de hipótesis para el zero-shot
if resolver_dispositivo() == "cuda":
    otra = zero_shot(cuerpos, etiquetas, plantilla="Este correo habla de {}.")
    print("F1 macro con otra plantilla:", f1_macro(motivos, [s["etiqueta"] for s in otra]), "y con la de la guía:", f1_zs)
else:
    print("Sección opcional omitida: sin GPU el zero-shot de 60 correos tarda unos 7 minutos (se usa la salida guardada).")

## Cierre

In [ ]:
# ✔ Cierre: los controles y el tiempo del cuaderno
esperados = ["C0", "C1", "C2", "C3", "C4", "C5", "C6", "C7", "C8", "C9"]
faltan = [c for c in esperados if c not in CONTROLES]
assert not faltan, f"Faltan controles por superar: {faltan}"
assert correr_pruebas(), "Alguna prueba de mi_pipeline falla."
print("✔ Controles superados:", ", ".join(CONTROLES))
print(f"Tiempo del cuaderno: {(time.perf_counter() - INICIO) / 60:.1f} minutos")
print("Siguiente: guardar el avance (commit) y subir la rama trabajo-s04, como indica la guía.")